# Deep Sets for joint $N_{\mathrm{part}}$ and $b$ regression

This notebook trains a permutation-invariant Deep Sets model directly on each UrQMD event's variable-length particle collection. Each particle is represented by its PDG identity and momentum-derived features; the network jointly predicts:

- participant count, $N_{\mathrm{part}}$
- impact parameter, $b$ [fm]

Only the first **10,000 raw events** are read, then split deterministically into 8,000 training, 1,000 validation, and 1,000 test events. The notebook is saved without execution outputs.

## 1. Imports and configuration

In [ ]:
from copy import deepcopy
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import torch
import uproot
from torch import nn
from torch.utils.data import DataLoader, Dataset

In [ ]:
SEED = 42
MAX_EVENTS = 10_000
N_TRAIN = 8_000
N_VAL = 1_000
N_TEST = 1_000
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-3
PID_EMBEDDING_DIM = 8
PHI_DIM = 64
PATIENCE = 6

assert N_TRAIN + N_VAL + N_TEST == MAX_EVENTS
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

## 2. Read the capped raw ROOT sample

The raw tree stores event-level `Npart` and `b`, plus variable-length `pid`, `px`, `py`, and `pz` particle arrays. Passing `entry_stop=MAX_EVENTS` prevents this notebook from reading or training on more than 10,000 events.

In [ ]:
relative_data_path = Path("Data/combined_100k.root")
data_path = next(
    (
        root / relative_data_path
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / relative_data_path).is_file()
    ),
    None,
)

if data_path is None:
    raise FileNotFoundError(f"Could not locate {relative_data_path} from the project tree.")

with uproot.open(data_path) as root_file:
    tree = root_file["urqmd"]
    if tree.num_entries < MAX_EVENTS:
        raise ValueError(f"ROOT tree contains only {tree.num_entries:,} events.")
    events = tree.arrays(
        ["pid", "px", "py", "pz", "Npart", "b"],
        entry_start=0,
        entry_stop=MAX_EVENTS,
        library="ak",
    )

assert len(events) == MAX_EVENTS
print(f"Loaded {len(events):,} events from {data_path}")
print("Particle count range:", int(ak.min(ak.num(events["pid"]))), "to", int(ak.max(ak.num(events["pid"]))))
print("Npart range:", int(ak.min(events["Npart"])), "to", int(ak.max(events["Npart"])))
print("b range:", float(ak.min(events["b"])), "to", float(ak.max(events["b"])), "fm")

## 3. Split events and define training-only metadata

The PID vocabulary and target normalization are derived from training events only. Index `0` is reserved for particle IDs not seen during training.

In [ ]:
permutation = rng.permutation(MAX_EVENTS)
train_idx = permutation[:N_TRAIN]
val_idx = permutation[N_TRAIN : N_TRAIN + N_VAL]
test_idx = permutation[N_TRAIN + N_VAL :]

assert len(np.unique(permutation)) == MAX_EVENTS

train_pids = ak.to_numpy(ak.flatten(events["pid"][train_idx]))
known_pids = np.unique(train_pids)
pid_to_index = {int(pid): index + 1 for index, pid in enumerate(known_pids)}
n_pid_tokens = len(pid_to_index) + 1

all_targets = np.column_stack(
    [
        ak.to_numpy(events["Npart"]).astype(np.float32),
        ak.to_numpy(events["b"]).astype(np.float32),
    ]
)
target_names = np.array(["Npart", "b [fm]"])
target_mean = all_targets[train_idx].mean(axis=0)
target_std = all_targets[train_idx].std(axis=0)
target_std = np.where(target_std > 1e-8, target_std, 1.0).astype(np.float32)

print(f"Train: {len(train_idx):,}; validation: {len(val_idx):,}; test: {len(test_idx):,}")
print(f"Training PID vocabulary: {len(known_pids)} species")
print("Training target mean:", dict(zip(target_names, target_mean.round(3))))
print("Training target std:", dict(zip(target_names, target_std.round(3))))

## 4. Build variable-length particle batches

Each particle receives five continuous features: $p_x$, $p_y$, $p_z$, $p_T$, and pseudorapidity $\eta$. PID is handled separately by a learned embedding. Events are batched without padding by concatenating their particles and carrying an event-index vector.

In [ ]:
class UrQMDEventDataset(Dataset):
    def __init__(self, events, indices, pid_to_index, target_mean, target_std):
        self.events = events
        self.indices = np.asarray(indices)
        self.pid_to_index = pid_to_index
        self.target_mean = np.asarray(target_mean, dtype=np.float32)
        self.target_std = np.asarray(target_std, dtype=np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        row = int(self.indices[item])
        pid = ak.to_numpy(self.events["pid"][row]).astype(np.int64)
        px = ak.to_numpy(self.events["px"][row]).astype(np.float32)
        py = ak.to_numpy(self.events["py"][row]).astype(np.float32)
        pz = ak.to_numpy(self.events["pz"][row]).astype(np.float32)

        pt = np.sqrt(px**2 + py**2)
        momentum = np.sqrt(px**2 + py**2 + pz**2)
        eta = np.arctanh(np.clip(pz / np.maximum(momentum, 1e-8), -0.999999, 0.999999))
        particle_features = np.column_stack([px, py, pz, pt, eta]).astype(np.float32)

        pid_tokens = np.fromiter(
            (self.pid_to_index.get(int(value), 0) for value in pid),
            dtype=np.int64,
            count=len(pid),
        )
        target = np.array(
            [self.events["Npart"][row], self.events["b"][row]],
            dtype=np.float32,
        )
        standardized_target = (target - self.target_mean) / self.target_std

        return (
            torch.from_numpy(particle_features),
            torch.from_numpy(pid_tokens),
            torch.from_numpy(standardized_target),
        )


def collate_events(batch):
    particle_features, pid_tokens, targets = zip(*batch)
    counts = torch.tensor([len(features) for features in particle_features], dtype=torch.long)
    event_index = torch.repeat_interleave(torch.arange(len(batch)), counts)

    return (
        torch.cat(particle_features, dim=0),
        torch.cat(pid_tokens, dim=0),
        event_index,
        torch.stack(targets),
    )

In [ ]:
train_dataset = UrQMDEventDataset(events, train_idx, pid_to_index, target_mean, target_std)
val_dataset = UrQMDEventDataset(events, val_idx, pid_to_index, target_mean, target_std)
test_dataset = UrQMDEventDataset(events, test_idx, pid_to_index, target_mean, target_std)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_events,
    generator=loader_generator,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_events,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_events,
    num_workers=0,
)

sample_features, sample_pids, sample_event_index, sample_targets = next(iter(train_loader))
print("Particles in one batch:", sample_features.shape)
print("Events in one batch:", sample_targets.shape)
print("PID tokens:", sample_pids.shape, "event indices:", sample_event_index.shape)

## 5. Define the Deep Sets model

The shared $\phi$ network embeds each particle. Mean pooling makes the representation invariant to particle order, while `log1p(multiplicity)` explicitly preserves event-size information. The event network $\rho$ maps this pooled representation to the two standardized targets.

In [ ]:
class DeepSetRegressor(nn.Module):
    def __init__(self, n_pid_tokens, pid_embedding_dim=8, phi_dim=64):
        super().__init__()
        self.pid_embedding = nn.Embedding(n_pid_tokens, pid_embedding_dim)
        self.phi = nn.Sequential(
            nn.Linear(5 + pid_embedding_dim, phi_dim),
            nn.ReLU(),
            nn.Linear(phi_dim, phi_dim),
            nn.ReLU(),
        )
        self.rho = nn.Sequential(
            nn.Linear(phi_dim + 1, phi_dim),
            nn.ReLU(),
            nn.Linear(phi_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, particle_features, pid_tokens, event_index, n_events):
        pid_embedding = self.pid_embedding(pid_tokens)
        particle_latent = self.phi(torch.cat([particle_features, pid_embedding], dim=1))

        pooled_sum = particle_latent.new_zeros((n_events, particle_latent.shape[1]))
        pooled_sum.index_add_(0, event_index, particle_latent)
        counts = torch.bincount(event_index, minlength=n_events).clamp_min(1).to(particle_latent.dtype)
        pooled_mean = pooled_sum / counts[:, None]
        event_features = torch.cat([pooled_mean, torch.log1p(counts)[:, None]], dim=1)
        return self.rho(event_features)


model = DeepSetRegressor(n_pid_tokens, PID_EMBEDDING_DIM, PHI_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
loss_fn = nn.MSELoss()

print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Train with validation-based early stopping

In [ ]:
def move_batch(batch):
    particle_features, pid_tokens, event_index, targets = batch
    return (
        particle_features.to(device),
        pid_tokens.to(device),
        event_index.to(device),
        targets.to(device),
    )


def epoch_loss(loader, training):
    model.train(training)
    total_loss = 0.0
    total_events = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            particle_features, pid_tokens, event_index, targets = move_batch(batch)

            if training:
                optimizer.zero_grad(set_to_none=True)

            predictions = model(
                particle_features,
                pid_tokens,
                event_index,
                n_events=len(targets),
            )
            loss = loss_fn(predictions, targets)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(targets)
            total_events += len(targets)

    return total_loss / total_events


history = {"train": [], "validation": []}
best_state = None
best_val_loss = np.inf
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = epoch_loss(train_loader, training=True)
    val_loss = epoch_loss(val_loader, training=False)
    history["train"].append(train_loss)
    history["validation"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f"Epoch {epoch:3d}: train={train_loss:.5f}, validation={val_loss:.5f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"Best standardized validation MSE: {best_val_loss:.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["train"], label="Train")
ax.plot(history["validation"], label="Validation")
ax.set(xlabel="Epoch", ylabel="Standardized MSE", title="Deep Sets training history")
ax.legend()
plt.show()

## 7. Evaluate each target in physical units

In [ ]:
def predict_physical(loader):
    model.eval()
    standardized_predictions = []
    standardized_targets = []

    with torch.no_grad():
        for batch in loader:
            particle_features, pid_tokens, event_index, targets = move_batch(batch)
            predictions = model(
                particle_features,
                pid_tokens,
                event_index,
                n_events=len(targets),
            )
            standardized_predictions.append(predictions.cpu().numpy())
            standardized_targets.append(targets.cpu().numpy())

    predictions = np.concatenate(standardized_predictions) * target_std + target_mean
    targets = np.concatenate(standardized_targets) * target_std + target_mean
    return targets, predictions


def metrics_by_target(y_true, y_pred):
    results = {}
    for column, name in enumerate(target_names):
        residual = y_pred[:, column] - y_true[:, column]
        mae = np.mean(np.abs(residual))
        rmse = np.sqrt(np.mean(residual**2))
        denominator = np.sum((y_true[:, column] - y_true[:, column].mean()) ** 2)
        r2 = 1.0 - np.sum(residual**2) / denominator
        results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    return results


val_true, val_pred = predict_physical(val_loader)
test_true, test_pred = predict_physical(test_loader)

for split_name, split_metrics in {
    "Validation": metrics_by_target(val_true, val_pred),
    "Test": metrics_by_target(test_true, test_pred),
}.items():
    print(split_name)
    for target_name, values in split_metrics.items():
        formatted = ", ".join(f"{key}={value:.4f}" for key, value in values.items())
        print(f"  {target_name}: {formatted}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for column, (ax, name) in enumerate(zip(axes, target_names)):
    ax.hexbin(val_true[:, column], val_pred[:, column], gridsize=35, mincnt=1, cmap="viridis")
    limits = [
        min(val_true[:, column].min(), val_pred[:, column].min()),
        max(val_true[:, column].max(), val_pred[:, column].max()),
    ]
    ax.plot(limits, limits, "r--", linewidth=1, label="Ideal")
    ax.set(xlabel=f"True {name}", ylabel=f"Predicted {name}", title=f"Validation: {name}")
    ax.legend()

fig.tight_layout()
plt.show()

## Interpretation notes

- Particle order is irrelevant: shuffling particles within an event leaves the pooled event representation unchanged.
- `b` is a regression target, never an input feature.
- PID vocabulary is fitted on training events only; unseen validation/test PIDs map to the unknown token.
- For a full study, compare sum, mean-plus-count, and attention pooling, and assess bias versus centrality and multiplicity.